# A Quick Recap for the each part of the AI agents field


##  LangChain vs LangGraph State & Component Comparison

This document summarizes the key concepts discussed: **LangChain**, **LangGraph**, **AgentState**, **TypedDict State**, **dataclass Context**, and **Pydantic BaseModel**.

---

### 1. High-Level Architecture

| Layer | Purpose | Typical Components | Who Owns It |
|------|---------|-------------------|-------------|
| Capability Layer | Provides AI functionality | LLMs, Tools, Prompts | LangChain |
| Execution Layer | Controls flow of execution | Graph nodes, edges, reducers | LangGraph |
| Agent Memory | Stores agent reasoning state | AgentState | LangChain |
| Workflow State | Data passed between steps | TypedDict State | LangGraph |
| Runtime Context | External configuration | dataclass context | Application |
| Data Contracts | Validate structured input/output | Pydantic BaseModel | Tools / APIs |

---

### 2. LangChain vs LangGraph

| Feature | LangChain | LangGraph |
|-------|-----------|-----------|
| Primary Role | AI capability framework | Execution orchestration |
| Core Concept | Agents | Graph workflows |
| Control Flow | Implicit agent loop | Explicit graph |
| State Management | Agent memory | Workflow state |
| Parallel Execution | Limited | Native |
| Determinism | Lower | High |
| Production Workflows | Harder to manage | Designed for production |
| Tooling | LLMs, prompts, tools | Nodes, edges, reducers |
| Interrupt Support | Agent-level | Graph-level |
| Checkpointing | Limited | Built-in |

---

### 3. AgentState vs TypedDict State

| Aspect | AgentState | TypedDict State |
|------|-------------|----------------|
| Library | LangChain | LangGraph |
| Purpose | Agent internal memory | Workflow shared state |
| Structure | Python class | Dictionary schema |
| Mutation Style | Mutable object | Functional updates |
| Scope | Single agent | Entire workflow |
| Supports Reducers | No | Yes |
| Parallel Updates | No | Yes |
| Context Growth | Continuous | Controlled |
| Used By | Tools, middleware | Graph nodes |
| Control Flow | Agent loop | Graph edges |

---

### 4. dataclass Context vs AgentState

| Aspect | dataclass Context | AgentState |
|------|------------------|-----------|
| Purpose | Runtime configuration | Agent memory |
| Mutability | Typically static | Mutable |
| Lifecycle | Provided at invocation | Changes during execution |
| Storage | Outside the agent | Inside the agent |
| Example Data | API keys, user info | authentication flags |
| Persistence | No | Optional |
| Context Window Impact | None | Yes |


### 5. TypedDict vs BaseModel

| Aspect | TypedDict | BaseModel |
|------|------------------|-----------|
| Library | Python typing | Pydantic |
| Purpose | Describe dictionary structure | Validate structured data |
| Runtime Validation | No | Yes |
| Used In | Graph state |Tools / APIs |
| Serialization | Manual | Automatic |
| Strict Typing | Static only | Runtime enforcedOptional |
| Ideal For | Workflow data flow | API schemas  |



### 6. State Evolution Comparison

| Property        | AgentState          | TypedDict         |
| --------------- | ------------------- | ----------------- |
| Memory Pattern  | Accumulating memory | Selective updates |
| Context Size    | Grows continuously  | Controlled        |
| Data Ownership  | Agent               | Workflow          |
| Update Method   | Attribute mutation  | Return dictionary |
| Parallel Safety | No                  | Yes               |


### 7. Mental Model Summary

| Concept           | Think of it as                       |
| ----------------- | ------------------------------------ |
| LangChain         | AI capability toolkit                |
| LangGraph         | Workflow execution engine            |
| AgentState        | What the agent remembers             |
| TypedDict State   | Data flowing through the workflow    |
| dataclass Context | Configuration provided to the system |
| BaseModel         | Contract enforcing structured data   |


### 8. Final Simplified Architecture Diagram
```text
Application
   |
   |-- Context (dataclass)
   |
LangGraph Workflow
   |
   |-- State (TypedDict)
   |
   |-- Node
        |
        |-- LangChain Agent
                |
                |-- AgentState
                |-- Tools
                |-- LLM
```

# Rag From Scratch: Routing

![alt text](<../../assets/Logical and Semantic routing.png>)

In [1]:
import warnings
import os 
from dotenv import load_dotenv

# 0. Disable Warnings
warnings.filterwarnings("ignore")

# 1. Add parentheses to actually run the function
load_dotenv()

try: 
    # 2. Use .get() with a default empty string "" to avoid NoneType errors
    os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGSMITH_TRACING_V2")
    os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
    os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")
    os.environ["LANGCHAIN_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT")
    os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY")
    os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
    os.environ["USER_AGENT"] = "MyLangChainApp/1.0" # For WebBaseLoader
    print("Environment variables set successfully")
except Exception as e: 
    print(f"Error: {e}")

Environment variables set successfully


### Part 10: Logical and Semantic routing
### First Logical Routing
Use function-calling for classification.

Flow:

![alt text](../../assets/Routing.png)

In [2]:
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_mistralai import ChatMistralAI


# Data Model 
class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""
    datasource: Literal["python_docs", "js_docs", "golang_docs"] = Field(
        ..., 
        description = "Given a user question choose which datasource would be most relevant for answering their question",
    )
    
# LLM with Function Call
llm = ChatMistralAI(model = "mistral-medium-latest",
                    temperature=0)
structured_llm = llm.with_structured_output(RouteQuery)

# Prompt
system = """You are an expert at routing a user question to the appropriate data source.

Based on the programming language the question is referring to, route it to the relevant data source."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

# Define Router
router = prompt | structured_llm

Note: we used function calling to produce structured output


![alt text](<../../assets/llm with function call.png>)

In [3]:
question =  """Why doesn't the following code work:

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
prompt.invoke("french")
"""

result = router.invoke({"question": question})

result

RouteQuery(datasource='python_docs')

In [4]:
result.datasource

'python_docs'

Once we have this, it is trivial to define a branch that uses result.datasource

In [5]:
from langchain_core.runnables import RunnableLambda


def choose_route(result):
    if "python_docs" in result.datasource.lower():
        # WE ADD THE LOGIC HERE IF ANY FOR Python Docs
        return "chain for python_docs"
    if "js_docs" in result.datasource.lower():
        # WE ADD THE LOGIC HERE IF ANY FOR JS Docs
        return "chain for js_docs"
    else:
        # WE ADD THE LOGIC HERE IF ANY FOR GoLang Docs
        return "chain for golang_docs"

full_chain  = router | RunnableLambda(choose_route)

full_chain.invoke({"question": question})


'chain for python_docs'

### Semantic routing

Flow: 


![alt text](<../../assets/Semantic Routing.png>)

In [ ]:
from langchain_community.utils.math import cosine_similarity
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings

# Two prompts
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""


# Embed prompts
embeddings = MistralAIEmbeddings(model = "mistral-embed")
prompt_template = [physics_template, math_template]
prompt_embeddings = embeddings.embed_documents(prompt_template)

# Route question to prompt
def prompt_router(input):
    # Embed Question 
    query_embedding = embeddings.embed_query(input["query"])
    
    # Compute Similarity 
    similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]
    most_similar = prompt_template[similarity.argmax()]
    
    # Chosen Prompt
    print("Using MATH" if most_similar == math_template else "Using PHYSICS")
    return PromptTemplate.from_template(most_similar)

chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | ChatMistralAI(model = "mistral-medium-latest", temperature = 0)
    | StrOutputParser()
)

print(chain.invoke("What's a black hole"))

Using PHYSICS
A **black hole** is a region in space where gravity is so strong that nothing—not even light—can escape from it. Here’s a concise breakdown:

1. **Formation**: Most black holes form when massive stars collapse under their own gravity at the end of their life cycles (supernova explosion). Supermassive black holes (millions to billions of times the Sun’s mass) likely form differently, often at the centers of galaxies.

2. **Key Features**:
   - **Event Horizon**: The "point of no return" around a black hole. Once crossed, escape is impossible.
   - **Singularity**: At the center, matter is crushed into an infinitely dense point (where our current physics breaks down).
   - **No-Hair Theorem**: Black holes are described by just three properties: **mass**, **electric charge**, and **spin** (angular momentum). All other details ("hair") are lost.

3. **Why "Black"?**: Light can’t escape, so they’re invisible. We detect them by observing their effects on nearby matter (e.g., ac

**`RunnableLambda`**: The Custom Worker
- Use RunnableLambda when you want to run your own custom Python code inside a chain. It turns a regular function into a "Runnable" that LangChain understands.

  - Main Use: Formatting text, cleaning data, or calling an external API that doesn't have a built-in LangChain tool.

  - Example: Changing a user's input to all lowercase before sending it to the AI.

**`RunnablePassthrough`**: The Conveyor Belt
- Use RunnablePassthrough to pass data from one step to the next without changing it. It is most commonly used to "carry" the user's original question forward while you are looking up information in a database.

  - Main Use: Keeping the original input available for later steps in the chain (often used with `.assign()`).

  - Example: If you need to search a database using a question, but you also need that same question later to generate the final answer.


| Tool | Analogy | Best For
| ----------------- | ---------------|--------------------- |
| RunnableLambda    | A custom machine |  Transforming data or running custom logic.
| RunnablePassthrough | A clear conveyor belt| Passing data forward unchanged or adding new keys.